In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
import warnings


def main():
    warnings.filterwarnings('ignore')

    #Data Read
    file_path = 'heart_disease_uci.csv'
    df = pd.read_csv(file_path, na_values='?')

    # Handle Missing Values
    for col in df.columns:
        if df[col].isnull().any():
            if df[col].dtype == 'object':
                mode_value = df[col].mode()[0]
                df[col].fillna(mode_value, inplace=True)
            else:
                mean_value = df[col].mean()
                df[col].fillna(mean_value, inplace=True)

    # Data Encoding
    df_processed = df.drop(['id', 'dataset'], axis=1)
    df_processed.rename(columns={'num': 'target'}, inplace=True)
    df_processed['target'] = (df_processed['target'] > 0).astype(int)
    categorical_cols = df_processed.select_dtypes(include=['object']).columns
    df_processed = pd.get_dummies(df_processed, columns=categorical_cols, drop_first=True)

    # Separate features (X) and target (y)
    X = df_processed.drop('target', axis=1)
    y = df_processed['target']

    #Split Data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)


    #Hyperparameter Tuning

    param_grid = {
        'C': [1, 5, 10, 15],          # Values around the previous best C=10
        'gamma': [0.1, 0.05, 0.01],   # Values around the previous best gamma=0.01
        'kernel': ['rbf']             # We know 'rbf' was the best kernel
    }

    grid_search = GridSearchCV(estimator=SVC(probability=True, random_state=42),
                               param_grid=param_grid,
                               cv=5,
                               verbose=2)
    grid_search.fit(X_train_scaled, y_train)


    #Best parameters
    print("Best Parameters Found:")
    print(grid_search.best_params_)
    print("\n")

    #Get the best estimator
    best_svm = grid_search.best_estimator_

    #Evaluate the Optimized Model
    print("-----------------------------------------")
    print("Evaluating the Optimized SVM Model...")
    print("-----------------------------------------")
    y_pred_optimized = best_svm.predict(X_test_scaled)
    optimized_accuracy = accuracy_score(y_test, y_pred_optimized)
    print(f"Optimized SVM Accuracy: {optimized_accuracy:.4f}\n")
    print("Classification Report for Optimized SVM:")
    print(classification_report(y_test, y_pred_optimized))

    #Comparison
    print("\n-----------------------------------------")
    print("Comparison: Baseline SVM Performance")
    print("-----------------------------------------")
    baseline_svm = SVC(probability=True, random_state=42)
    baseline_svm.fit(X_train_scaled, y_train)
    y_pred_baseline = baseline_svm.predict(X_test_scaled)
    baseline_accuracy = accuracy_score(y_test, y_pred_baseline)
    print(f"Baseline SVM Accuracy: {baseline_accuracy:.4f}\n")
    print("Classification Report for Baseline SVM:")
    print(classification_report(y_test, y_pred_baseline))


if __name__ == "__main__":
    main()



Fitting 5 folds for each of 12 candidates, totalling 60 fits
[CV] END .........................C=1, gamma=0.1, kernel=rbf; total time=   0.0s
[CV] END .........................C=1, gamma=0.1, kernel=rbf; total time=   0.0s
[CV] END .........................C=1, gamma=0.1, kernel=rbf; total time=   0.0s
[CV] END .........................C=1, gamma=0.1, kernel=rbf; total time=   0.0s
[CV] END .........................C=1, gamma=0.1, kernel=rbf; total time=   0.0s
[CV] END ........................C=1, gamma=0.05, kernel=rbf; total time=   0.0s
[CV] END ........................C=1, gamma=0.05, kernel=rbf; total time=   0.0s
[CV] END ........................C=1, gamma=0.05, kernel=rbf; total time=   0.0s
[CV] END ........................C=1, gamma=0.05, kernel=rbf; total time=   0.0s
[CV] END ........................C=1, gamma=0.05, kernel=rbf; total time=   0.0s
[CV] END ........................C=1, gamma=0.01, kernel=rbf; total time=   0.0s
[CV] END ........................C=1, gamma=0.01